# LABORATÓRIO 10: O Pipeline Definitivo
## RAG + QLoRA + FlashAttention + KV Cache

> **Partes deste laboratório foram geradas/complementadas com IA, revisadas e validadas por [Seu Nome]**

**Objetivo:** Orquestrar um pipeline de IA ponta a ponta simulando um ambiente de produção HealthTech, resolvendo o problema de Out-Of-Memory (OOM) em GPU com QLoRA, KV Cache e FlashAttention-2.

## 0. Instalação das Dependências

In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q flash-attn --no-build-isolation
!pip install -q torch torchvision

## 1. Imports e Verificação de Hardware

In [ ]:
import torch
import time
import random
import string
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Verificação de GPU ──────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise EnvironmentError(
        "GPU não detectada. Este laboratório requer uma GPU CUDA. "
        "Use Google Colab (Runtime > Change runtime type > T4 GPU) ou "
        "um ambiente com GPU disponível."
    )

device = "cuda"
gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

print(f"✅ GPU detectada : {gpu_name}")
print(f"   VRAM total    : {total_vram:.0f} MB")
print(f"   PyTorch versão: {torch.__version__}")

## Passo 1 – Ingestão Eficiente: Carregamento QLoRA em 4-bits

Ao invés de carregar o modelo em Float16 (que ocuparia ~2 GB de VRAM para um 1.1B),
usamos **bitsandbytes** com `load_in_4bit=True`. Isso reduz o uso de memória em ~4×.

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ── Configuração QLoRA 4-bits ───────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,      # double quantization → ainda menos memória
    bnb_4bit_quant_type="nf4",           # NormalFloat4: melhor para pesos LLM
)

print("⏳ Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# ── Zerar contadores de memória antes do carregamento ──────────────────────
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()
mem_before_load = torch.cuda.memory_allocated() / (1024 ** 2)

print("⏳ Carregando modelo em 4-bits (QLoRA)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation será adicionado no Passo 4
)
model.eval()

mem_after_load = torch.cuda.memory_allocated() / (1024 ** 2)
vram_modelo_mb = mem_after_load - mem_before_load

print(f"\n{'='*55}")
print(f"  📊 MÉTRICA – PASSO 1: VRAM após carregamento 4-bits")
print(f"{'='*55}")
print(f"  VRAM antes do load : {mem_before_load:.1f} MB")
print(f"  VRAM após o load   : {mem_after_load:.1f} MB")
print(f"  ✅ Modelo ocupa    : {vram_modelo_mb:.1f} MB (4-bits QLoRA)")
print(f"{'='*55}")

## Passo 2 – Simulando o RAG Massivo

Geramos um texto fictício de **~12.000 tokens** simulando 5 capítulos de manuais médicos
recuperados pelo banco vetorial (como feito no Lab 09 com FAISS/ChromaDB).

In [ ]:
# ── Corpus médico sintético ────────────────────────────────────────────────
CAPITULOS_MEDICOS = [
    ("Capítulo 1: Farmacologia Clínica e Interações Medicamentosas",
     "A farmacologia clínica estuda os efeitos dos fármacos no organismo humano. "
     "As interações medicamentosas ocorrem quando dois ou mais fármacos administrados "
     "simultaneamente alteram mutuamente seus efeitos farmacológicos. "
     "A warfarina, anticoagulante oral, tem seu metabolismo hepático inibido por "
     "fluconazol, aumentando significativamente o risco hemorrágico. "
     "Os inibidores da bomba de prótons, como omeprazol e pantoprazol, reduzem a "
     "ativação do clopidogrel, comprometendo sua ação antiagregante plaquetária. "
     "A digoxina tem janela terapêutica estreita; seu nível sérico é elevado por "
     "amiodarona e claritromicina através da inibição da glicoproteína-P renal. "
     "O monitoramento terapêutico de fármacos é essencial em pacientes polimedicados, "
     "especialmente idosos com múltiplas comorbidades e função renal comprometida. "
     "A biotransformação hepática envolve reações de fase I (oxidação, redução, hidrólise) "
     "mediadas pelo sistema citocromo P450 e reações de fase II (conjugação). "),

    ("Capítulo 2: Semiologia Cardiovascular e Exames Diagnósticos",
     "A semiologia cardiovascular compreende a anamnese, exame físico e exames complementares. "
     "O eletrocardiograma de 12 derivações permanece o exame inicial fundamental na avaliação "
     "de dor torácica aguda, identificando supra desnivelamento do segmento ST na síndrome "
     "coronariana aguda com supra de ST (IAMCSST). "
     "O ecocardiograma transtorácico avalia função sistólica (fração de ejeção), função "
     "diastólica, valvopatias e derrame pericárdico. "
     "A pressão arterial deve ser medida com técnica correta: paciente sentado há 5 minutos, "
     "braço direito apoiado ao nível do coração, manguito adequado ao perímetro braquial. "
     "Biomarcadores cardíacos: troponina I e T ultrassensíveis elevam-se 2-4 horas após "
     "isquemia miocárdica, com pico em 12-24h e normalização em 5-10 dias. "
     "O BNP e NT-proBNP são marcadores de disfunção ventricular e insuficiência cardíaca. "),

    ("Capítulo 3: Urgências em Neurologia – AVC e Epilepsia",
     "O acidente vascular cerebral isquêmico (AVCi) é emergência neurológica com janela "
     "terapêutica estreita para trombólise intravenosa com alteplase (até 4,5 horas) e "
     "trombectomia mecânica (até 24 horas em casos selecionados por neuroimagem). "
     "O protocolo FAST (Face, Arm, Speech, Time) orienta o reconhecimento precoce do AVC. "
     "A tomografia computadorizada sem contraste é o exame inicial para excluir hemorragia. "
     "A ressonância magnética com difusão detecta isquemia aguda precocemente. "
     "O status epilepticus é definido como convulsão com duração superior a 5 minutos ou "
     "duas crises sem recuperação da consciência entre elas. "
     "Tratamento: benzodiazepínico IV (diazepam ou lorazepam) como primeira linha, "
     "seguido de fenitoína ou ácido valproico IV como segunda linha terapêutica. "),

    ("Capítulo 4: Pneumologia – Insuficiência Respiratória e Ventilação Mecânica",
     "A insuficiência respiratória aguda tipo I (hipoxêmica) caracteriza-se por PaO2 < 60mmHg "
     "com PaCO2 normal ou reduzida, causada por shunt intrapulmonar (pneumonia, SDRA, edema). "
     "A tipo II (hipercápnica) apresenta PaCO2 > 45mmHg, decorrente de hipoventilação alveolar. "
     "A oxigenoterapia deve ser titulada para SpO2 entre 94-98% (88-92% em pacientes com "
     "DPOC e risco de hipercapnia). "
     "A ventilação mecânica protetora na SDRA preconiza volume corrente de 6 ml/kg de peso "
     "predito, pressão de platô ≤ 30 cmH2O e PEEP adequada para recrutamento alveolar. "
     "A traqueostomia precoce (7-14 dias) reduz dias de ventilação mecânica e complicações "
     "em pacientes com perspectiva de ventilação prolongada. "),

    ("Capítulo 5: Endocrinologia – Diabetes Mellitus e Emergências Hiperglicêmicas",
     "O diabetes mellitus tipo 2 resulta de resistência insulínica progressiva associada a "
     "disfunção secretória das células beta pancreáticas. "
     "Metas glicêmicas: HbA1c < 7% para a maioria dos adultos, com individualização conforme "
     "idade, comorbidades, risco hipoglicêmico e expectativa de vida. "
     "Cetoacidose diabética (CAD): glicemia > 250 mg/dL, pH < 7,3, bicarbonato < 15 mEq/L "
     "e cetonemia/cetonúria. Tratamento: hidratação vigorosa com SF 0,9%, insulinoterapia "
     "contínua IV e reposição de potássio (não iniciar insulina se K+ < 3,5 mEq/L). "
     "Estado hiperosmolar hiperglicêmico (EHH): glicemia > 600 mg/dL, osmolaridade sérica "
     "> 320 mOsm/kg, ausência de cetose significativa, mais comum em DM tipo 2 em idosos. "
     "O hipoglicemia grave (glicemia < 54 mg/dL com sintomas neuroglicopênicos) é tratada "
     "com 20-40 mL de glicose 50% IV ou glucagon IM/SC se acesso venoso indisponível. "),
]

# ── Montar o contexto RAG simulado ────────────────────────────────────────
def gerar_contexto_rag(capitulos: list, repeticoes_por_cap: int = 6) -> str:
    """Repete os capítulos para atingir ~12.000 tokens de contexto."""
    partes = []
    for titulo, conteudo in capitulos:
        partes.append(f"\n\n### {titulo}\n")
        partes.append(conteudo * repeticoes_por_cap)
    return "".join(partes)

contexto_rag = gerar_contexto_rag(CAPITULOS_MEDICOS, repeticoes_por_cap=8)

# ── Tokenizar e contar ────────────────────────────────────────────────────
tokens_contexto = tokenizer(
    contexto_rag,
    return_tensors="pt",
    truncation=True,
    max_length=12000,
)
n_tokens = tokens_contexto["input_ids"].shape[1]

print(f"📄 Contexto RAG gerado com {len(contexto_rag):,} caracteres")
print(f"🔢 Total de tokens tokenizados: {n_tokens:,}")

## Passo 3 – O Gargalo: Geração SEM KV Cache

Com `use_cache=False`, o modelo **recalcula Q, K e V do zero** a cada novo token gerado.
Isso resulta em complexidade **O(n²)** por token e uso massivo de VRAM.

In [ ]:
# ── Construir o prompt completo ───────────────────────────────────────────
INSTRUCAO = (
    "Com base nos manuais médicos acima, gere um resumo clínico estruturado "
    "destacando as principais emergências, fármacos e condutas descritas."
)

prompt_completo = (
    f"<|system|>\nVocê é um assistente médico especializado em resumos clínicos.</s>\n"
    f"<|user|>\n{contexto_rag[:8000]}\n\n{INSTRUCAO}</s>\n"
    f"<|assistant|>\n"
)

input_ids = tokenizer(
    prompt_completo,
    return_tensors="pt",
    truncation=True,
    max_length=10000,
).input_ids.to(device)

n_tokens_prompt = input_ids.shape[1]
print(f"📥 Tokens no prompt de entrada: {n_tokens_prompt:,}")

# ── Geração SEM KV Cache ──────────────────────────────────────────────────
N_NOVOS_TOKENS = 100

model.config.use_cache = False          # ← A PEGADINHA: desabilita KV Cache

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print("\n⚠️  Gerando tokens SEM KV Cache (lento e pesado em VRAM)...")
t_inicio = time.time()

with torch.no_grad():
    output_sem_cache = model.generate(
        input_ids,
        max_new_tokens=N_NOVOS_TOKENS,
        do_sample=False,                 # greedy decoding para reprodutibilidade
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

t_sem_cache = time.time() - t_inicio
vram_peak_sem_cache = torch.cuda.max_memory_allocated() / (1024 ** 2)

tokens_gerados = output_sem_cache[0][n_tokens_prompt:]
texto_gerado_sem_cache = tokenizer.decode(tokens_gerados, skip_special_tokens=True)

print(f"\n{'='*55}")
print(f"  📊 MÉTRICA – PASSO 3: SEM KV Cache")
print(f"{'='*55}")
print(f"  ⏱️  Tempo total de geração : {t_sem_cache:.2f} s")
print(f"  🔥 Pico de VRAM           : {vram_peak_sem_cache:.1f} MB")
print(f"  📝 Tokens gerados         : {len(tokens_gerados)}")
print(f"{'='*55}")
print(f"\n📋 Texto gerado (sem cache):\n{texto_gerado_sem_cache}")

## Passo 4 – Engenharia de Otimização: KV Cache + FlashAttention-2

### Por que funciona?
- **KV Cache**: armazena as matrizes K e V já computadas; cada novo token só precisa calcular **sua própria** linha de atenção.
- **FlashAttention-2**: reescreve o kernel de atenção para operar diretamente na SRAM da GPU, evitando idas e vindas à DRAM (HBM) — operação *IO-Aware*.

In [ ]:
# ── Recarregar modelo com FlashAttention-2 ────────────────────────────────
print("⏳ Recarregando modelo com FlashAttention-2 + QLoRA 4-bits...")

del model          # libera VRAM
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2",   # ← FlashAttention-2 ativado
)
model.eval()
model.config.use_cache = True                  # ← KV Cache ativado

print("✅ Modelo recarregado com otimizações.")

# ── Geração COM KV Cache + FlashAttention-2 ───────────────────────────────
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print("\n✅ Gerando tokens COM KV Cache + FlashAttention-2...")
t_inicio_opt = time.time()

with torch.no_grad():
    output_com_cache = model.generate(
        input_ids,
        max_new_tokens=N_NOVOS_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

t_com_cache = time.time() - t_inicio_opt
vram_peak_com_cache = torch.cuda.max_memory_allocated() / (1024 ** 2)

tokens_gerados_opt = output_com_cache[0][n_tokens_prompt:]
texto_gerado_com_cache = tokenizer.decode(tokens_gerados_opt, skip_special_tokens=True)

print(f"\n{'='*55}")
print(f"  📊 MÉTRICA – PASSO 4: COM KV Cache + FlashAttention-2")
print(f"{'='*55}")
print(f"  ⏱️  Tempo total de geração : {t_com_cache:.2f} s")
print(f"  🔥 Pico de VRAM           : {vram_peak_com_cache:.1f} MB")
print(f"  📝 Tokens gerados         : {len(tokens_gerados_opt)}")
print(f"{'='*55}")
print(f"\n📋 Texto gerado (com cache + FA2):\n{texto_gerado_com_cache}")

## Comparativo Final de Métricas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

labels     = ["Sem Cache\n(Passo 3)", "KV Cache +\nFlashAttention-2\n(Passo 4)"]
tempos     = [t_sem_cache, t_com_cache]
vrams      = [vram_peak_sem_cache, vram_peak_com_cache]
cores_t    = ["#e74c3c", "#2ecc71"]
cores_v    = ["#e67e22", "#3498db"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Lab 10 – Benchmark: Sem vs Com Otimizações de Inferência",
             fontsize=14, fontweight="bold")

# Gráfico 1 – Tempo
bars1 = axes[0].bar(labels, tempos, color=cores_t, edgecolor="black", width=0.4)
axes[0].set_title("Tempo de Geração (100 tokens)", fontsize=12)
axes[0].set_ylabel("Segundos (s)")
axes[0].set_ylim(0, max(tempos) * 1.3)
for bar, val in zip(bars1, tempos):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}s", ha="center", va="bottom", fontweight="bold")

# Gráfico 2 – VRAM
bars2 = axes[1].bar(labels, vrams, color=cores_v, edgecolor="black", width=0.4)
axes[1].set_title("Pico de VRAM durante Geração", fontsize=12)
axes[1].set_ylabel("Megabytes (MB)")
axes[1].set_ylim(0, max(vrams) * 1.3)
for bar, val in zip(bars2, vrams):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f"{val:.0f} MB", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig("benchmark_lab10.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Tabela resumo ─────────────────────────────────────────────────────────
speedup   = t_sem_cache / t_com_cache if t_com_cache > 0 else float("inf")
reducao_v = (1 - vram_peak_com_cache / vram_peak_sem_cache) * 100 if vram_peak_sem_cache > 0 else 0

print("\n" + "="*55)
print("  RESUMO COMPARATIVO")
print("="*55)
print(f"  {'Configuração':<30} {'Tempo':>8} {'VRAM Pico':>12}")
print("-"*55)
print(f"  {'Sem KV Cache':.<30} {t_sem_cache:>7.1f}s {vram_peak_sem_cache:>10.0f} MB")
print(f"  {'KV Cache + FlashAttn-2':.<30} {t_com_cache:>7.1f}s {vram_peak_com_cache:>10.0f} MB")
print("-"*55)
print(f"  Speedup de tempo    : {speedup:.1f}×  mais rápido")
print(f"  Redução de VRAM     : {reducao_v:.1f}%  menos memória")
print("="*55)